<a href="https://colab.research.google.com/github/Williams-MP/Algoritmos_de_optimizacion_entregas/blob/main/Seminario_Algoritmos_Angel_Encinas_%26_Williams_M%C3%A9rida.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Algoritmos de optimización - Seminario<br>
Nombre y Apellidos:  Ángel Encinas Moreno & Williams Mérida Parard   <br>
Url:https://github.com/Williams-MP/Algoritmos_de_optimizacion_entregas/tree/9086b65de7cbc459251d3467e8d2877842f62392/Seminario <br>
Problema:
> 1. Sesiones de doblaje <br>
>2. Organizar los horarios de partidos de La Liga<br>
>3. Combinar cifras y operaciones

**Descripción del problema:**

Se precisa coordinar el doblaje de una película. Los actores del doblaje deben coincidir en las tomas en las que sus personajes aparecen juntos en las diferentes tomas. Los actores de doblaje cobran todos la misma cantidad por cada día que deben desplazarse hasta el estudio de grabación independientemente del número de tomas que se graben. No es posible grabar más de 6 tomas por día. El objetivo es planificar las sesiones por día de manera que el gasto por los servicios de los actores de doblaje sea el menor posible. Los datos son:

Número de actores:  10
Número de tomas :   30
Actores/Tomas : https://bit.ly/36D8IuK

(*) La respuesta es obligatoria
                                        

## Importación de librerias necesarias

In [58]:
import pandas as pd
import numpy as np
import time
import math
import random
import itertools

## Carga de datos

In [59]:
import pandas as pd
import numpy as np

# Cargamos el archivo excel en la misma ruta.
nombre_archivo = "Datos problema doblaje(30 tomas, 10 actores).xlsx"
# Ajusta el skiprows o usecols si tu Excel tiene cabeceras raras
df_completo = pd.read_excel(nombre_archivo)

# Seleccionar los datos numéricos y evitamos las cabeceras
datos_numericos = df_completo.iloc[1:31, 1:11].dropna().astype(int)

# Creamos un dataframe con los datos
registro_sesiones = pd.DataFrame(
    datos_numericos.values, 
    index=np.arange(len(datos_numericos)), 
    columns=['Actor ' + str(x) for x in np.arange(1, 11)]
)
registro_sesiones.index.name = 'Toma -1 '

# Calculamos el total, con esto aseguramos cualquier discrepancia con el total del excel.
registro_sesiones['Total'] = np.sum(registro_sesiones, axis=1)

# Visualizamos los resultados
display(registro_sesiones)

# Asignamos el DF a la variable data.
data = registro_sesiones

,Actor 1,Actor 2,Actor 3,Actor 4,Actor 5,Actor 6,Actor 7,Actor 8,Actor 9,Actor 10,Total
Toma -1,,,,,,,,,,,
0,1,1,1,1,1,0,0,0,0,0,5
1,0,0,1,1,1,0,0,0,0,0,3
2,0,1,0,0,1,0,1,0,0,0,3
3,1,1,0,0,0,0,1,1,0,0,4
4,0,1,0,1,0,0,0,1,0,0,3
5,1,1,0,1,1,0,0,0,0,0,4
6,1,1,0,1,1,0,0,0,0,0,4
7,1,1,0,0,0,1,0,0,0,0,3
8,1,1,0,1,0,0,0,0,0,0,3


## ¿Cuantas posibilidades hay sin tener en cuenta las restricciones?<br>

Si no tenemos en cuenta las restricciones de capacidad y tiempo se podría asegurar que las posibilidades serían virtualmente infinitas. Al no existir un número mínimo de tomas por día, ní un máximo de días para finalizar la grabación, el número de calendarios de rodaje no tiene un límite acotado, ya que se podría extender la producción indefinidamente sin llegar a completar nunca las grabaciones.

---

## ¿Cuantas posibilidades hay teniendo en cuenta todas las restricciones.<br>

A continuación de consideran las restricciones:
- **Número de actores:** 10
- **Número de tomas:** 30
    - En caso que no tener mas restricciones que este, las 30 tomas se podrían repartir en 30 días. El número de combinaciones ascendería a $$30^{30} = 205891132094649000000000000000000000000000000$$
- **Límite de producción:** No es posible grabar más de 6 tomas por día.
    - Con esta nueva restricción podemos deducir que el menor tiempo posible. Eso resultaría en $$30 \text{ tomas} / 6 \text{ tomas por día} = 5 \text{ días}$$
- **Sincronización**: Los actores de deben de coincidir en las tomas en las que sus personajes aparecen juntos.
- **Optimización de costes**: Gasto por los servicios de los actores ha de ser el menor posible. Esto quiere decir, que cada día han de grabarse cuantas más tomas mejor (no más de 6).
- **Política de pago:** Cada actor cobra por el día que se desplaza al estudio, independiente del número de tomas a grabar.
- **Cálculo de posibilidades**
    - Teniendo en cuenta las restricciones anteriores, se debería de asignar a cada toma un día concreto, tratando de llenar siempre la cuata máxima de 6 tomas al día. Además, considerando que los actores ya están allí, entonces se podría reducir el problema a organizar las 30 tomas en 5 bloques de6.

A={1,1,1,1,1,1,2,2,2,2,2,2,3,3,3,3,3,3,4,4,4,4,4,4,5,5,5,5,5,5} 

Las soluciones posibles para esta organización se calcularía mediante la fórmula de permutaciones con repetición sobre el conjunto $A$, lo que nos da un número total de combinaciones de:

$$P_{30}^{6,6,6,6,6} = \frac{30!}{6! \cdot 6! \cdot 6! \cdot 6! \cdot 6!} = \frac{30!}{(720)^5}$$$$P = 1370874167589326400$$

---

## Modelo para el espacio de soluciones<br>
### (*) ¿Cual es la estructura de datos que mejor se adapta al problema? Argumentalo.(Es posible que hayas elegido una al principio y veas la necesidad de cambiar, arguentalo)


- Para representar esto de la mejor forma, la estructura de datos más adecuada es una lista de números enteros de tamaño 30.Cada posición $n$ de la lista corresponde a la toma $n$ de nuestra tabla de datos, y el número almacenado en esa posición representa el día que se le ha asignado.
- Siguiendo las restricciones mencionadas, cualquier lista que sea una solución válida será una permutación del conjunto $A$. De esta manera, nos aseguramos de que cada día (del 1 al 5) aparezca exactamente 6 veces en la lista, respetando el límite de capacidad y cubriendo todas las tomas.

---

## Según el modelo para el espacio de soluciones<br>
### (*)¿Cual es la función objetivo?

- Se puede decir que la función objetivo es aquella que calcula el número de desplazamientos a la grabación de los actores. Buscamos contabilizar el número de vecesque aparece cada actor de doblaje a lo largo de todos los días de grabación.
- Para obtener la función hay que agrupar las tomas asignadas a cada día.
    - Cada toma se representa con dos valores (binario), donde 1 y 0 representan "asistir" y "no asistir" respectivamente.
    -  Debido a que un actor ha de ir al estudio independientemente de si graba una toma **o** más de 1, entonces se ha de incluir la función lógica **OR**. Esta se puede representar con $|$.
    -  Finalmente, deberíamos contar todos los actores que han ido ese día al estudio. Es decir, sumanos los actores. Sería equivalente a la función count().
    -  Con esta información podemos decir que función objetivo sería algo como: $$\sum_{j=1}^{M} count(T_{1j}| T_{2j}| ...| T_{kj})$$ donde: $j = \text{índice del día}$, $ M: \text{total de días de duración de la grabación}$, $$T_{ij}$$ donde $i = \text{el número de toma}$ y $j = \text{el día asignado}$

### (*)¿Es un problema de maximización o minimización?

Es un problema de minimización. Se intenta reducir al máximo la cantidad de veces que los actores van el estudio, agrupando las tomas de forma que los repartos coincidan lo más posible, y de esta manera reducir costes.

### Ejemplo

Vamos a usar un ejemplo para que se vea la idea.Imaginemos que tenemos 4 tomas y 2 días disponibles (máximo 2 tomas por día).

- **Solución 1**: [1,1,2,2]
Agrupamos las tomas 1 y 2 el Día 1, y las tomas 3 y 4 el Día 2.
    - Día 1 (Tomas 1 y 2): Imaginemos que el OR lógico da 5 actores.
    - Día 2 (Tomas 3 y 4): Imaginemos que el OR lógico da 4 actores.
    - Total pagas: $5 + 4 = \mathbf{9}$

- **Solución 2**: [1,2,1,2]
Agrupamos las tomas 1 y 3 el Día 1, y las tomas 2 y 4 el Día 2.
    - Día 1 (Tomas 1 y 3): Imaginemos que el OR lógico da 6 actores.
    - Día 2 (Tomas 2 y 4): Imaginemos que el OR lógico da 6 actores.
    - Total pagas: $6 + 6 = \mathbf{12}$

- **Conclusión:**
    - La Solución 1 vuelve a ser mejor ($9$ que pagamos contra $12$). Se peude ver que al mezclar tomas que no comparten actores (Solución 2), aumentamos innecesariamente el número de personas que deben desplazarse al estudio cada día.
 
---

Para encontrar la mejor combinación de rodaje entre todas las posibilidades, primero necesitamos una herramienta matemática que nos diga exactamente cuánto cuesta cualquier plan que se nos ocurra. La función **calcular_coste_rodaje** de es nuestro motor de evaluación: toma un plan de trabajo, aplica la lógica de asistencia de los actores y nos devuelve el gasto total en dinero.

In [60]:
# Es función lo que hace es sumar cuánto nos va a costar el rodaje
# según cómo hayamos repartido las tomas en los días.
def calcular_coste_rodaje(plan_rodaje, matriz_asistencia, verbose=True):
  try:
    # Primero comprobamos que hayamos asignado un día a cada toma
    assert len(plan_rodaje) == len(matriz_asistencia)
  except:
    print('CUIDADO: Las tomas y el plan de días no coinciden')
    return float('inf')

  # Hacemos una copia de los datos por seguridad
  datos_con_dia = matriz_asistencia.copy()
  # Añadimos una columna nueva para saber en qué día se graba cada toma
  datos_con_dia['DiaAsignado'] = pd.Series(plan_rodaje)

  total_pagas = 0
  # Vamos recorriendo cada día de rodaje
  for dia_actual in np.unique(plan_rodaje):
    # Seleccionamos solo las tomas que tocan ese día
    tomas_del_dia = datos_con_dia[datos_con_dia['DiaAsignado'] == dia_actual]
    
    # tomas_del_dia.iloc[:,:-2] --> cogemos los 1s y 0s de los actores
    # np.any(, axis=0), --> Si un actor sale en alguna toma del día se marca "True".
    # 3. np.sum() --> Contamos cuántos actores tienen que venir, o sea los "Trues"
    
    actores_que_vienen = np.sum(np.any(tomas_del_dia.iloc[:,:-2], axis=0))
    total_pagas += actores_que_vienen

  if verbose: print(f'Total pagas con este plan {plan_rodaje}: {total_pagas}')
  return total_pagas


---

## Diseña un algoritmo para resolver el problema por fuerza bruta

La estrategia aquí es simple pero costosa: generamos todas las combinaciones posibles de tomas por día, calculamos el coste de cada una y nos quedamos con la más barata. Para esto, necesitamos crear una función que primero calcule todas las permutaciones y luego vaya probando una a una hasta dar con la solución óptima.

In [44]:
def fuerza_bruta(matriz_asistencia, max_tomas_dia):
  total_tomas = len(matriz_asistencia)
  # Calculamos el mínimo de días teóricos
  dias_minimos = math.ceil(total_tomas / max_tomas_dia)
  
  # Pintamos un poco de info inicial
  print(f'{len(matriz_asistencia.loc[0])-1} Actores')
  print(f'{total_tomas} Tomas')
  print(f'Máximo {max_tomas_dia} tomas por día, mínimo {dias_minimos} días\n')

  # Creamos una lista base repartiendo las tomas entre los días mínimos
  # Ejemplo con 6 tomas y 3 por día -> [1, 1, 1, 2, 2, 2]
  plantilla_base = [dia for dia in range(1, dias_minimos + 1) for _ in range(max_tomas_dia)]
  plantilla_base = plantilla_base[:total_tomas]
  
  # Generamos TODAS las permutaciones (esto es lo que tarda tanto)
  todas_las_combinaciones = itertools.permutations(plantilla_base)

  coste_minimo = float('inf')
  mejor_plan = []
  
  # Vamos probando cada combinación una por una
  for plan in todas_las_combinaciones:
    # Calculamos cuánto cuesta este plan usando la función que hicimos antes
    coste_actual = calcular_coste_rodaje(plan, matriz_asistencia, verbose=False)
    
    # Si este plan es más barato que el anterior más barato, nos lo quedamos
    if coste_actual < coste_minimo:
      coste_minimo = coste_actual
      mejor_plan = plan

  print(f'Mejor asignación: {mejor_plan}, con coste {coste_minimo}')
  return mejor_plan, coste_minimo

### Calcula la complejidad del algoritmo por fuerza bruta

La complejidad es: $$\frac{n!}{k!^M}$$ 
Este es principal inconveniente de este método, donde el número de combinaciones puede incrementar muy rápidadmente. Como esto crece a un ritmo de factorial $O(n!)$ . Para nuestro caso de 30 tomas, no será posible probar todas las posibilidades, a no ser que lo hagamos por computación cuántica.

**IMPORTANTE** Debido a que la complejidad es factorial, no será posible obtener un resultado usando 30 registros. Para comprobar que la función trabaja correctamente, reduciré a 5 registros (tomas). 

In [49]:
# Creamos una variable 'data_para_fuerza_bruta' con solo 5 tomas
# Para comprobar que el código funciona con menos registros.
# Seleccionamos las primeras 5 filas (iloc[0:5])
data_para_fuerza_bruta = data.iloc[0:5].copy()

In [50]:
# Ponemos el cronómetro en marcha
inicio_cronometro = time.time()

# Ejecutamos la búsqueda de la mejor solución (fuerza bruta)
_, mejor_coste = fuerza_bruta(data_para_fuerza_bruta, 6)

# Calculamos cuánto ha tardado y mostramos el resultado
tiempo_total = round(time.time() - inicio_cronometro, 2)
print(f'Búsqueda Óptima: coste {mejor_coste} en {tiempo_total} segundos')

10 Actores
5 Tomas
Máximo 6 tomas por día, mínimo 1 días

Mejor asignación: (1, 1, 1, 1, 1), con coste 7
Búsqueda Óptima: coste 7 en 0.19 segundos


---

## (*)Diseña un algoritmo que mejore la complejidad del algortimo por fuerza bruta. Argumenta porque crees que mejora el algoritmo por fuerza bruta

### Voraz

Tras comprobar que la fuerza bruta es inviable ($O(n!)$), una posible alternativa es implementar un algoritmo voraz. Esta técnica es ideal cuando buscamos eficiencia y el espacio de soluciones es muy grande.
#### ¿Por qué elegir esta técnica? 
- Según la definición de clase, la técnica voraz se basa en dividir el problema por etapas, tomando en cada una la decisión más ambiciosa y óptima sin mirar atrás.
    - **Eficiencia**: Es la técnica más recomendable por su rapidez.
    - **Aplicabilidad**: Cumple con las características necesarias: tenemos candidatos (tomas), una función de selección (agrupar por proximidad de actores), y una función de factibilidad (máximo $k$ tomas por día).

#### Funcionamiento Aplicado al Problema:
- Selección: Localizamos la toma con más actores y asignamos el día.
- Proximidad: Calculamos cuántos actores comparten las demás tomas con la elegida.
- Ambición: Agrupamos las tomas con mayor proximidad para minimizar costes inmediatamente.
Conclusión:
Aunque voraz no garantiza el óptimo global en todos los casos, nos proporciona una solución factible en un timepo menor, superando el mayor inconveniente de la fuerza bruta.

In [86]:
# Este algoritmo construye la solución día a día, tomando la mejor decisión 
# local posible en cada paso sin mirar al futuro. Es super rápido.
def alg_voraz(matriz_asistencia, max_tomas_dia, verbose=True):
    # Trabajamos sobre una copia para no romper los datos originales
    datos_trabajo = matriz_asistencia.copy()
    total_tomas = len(datos_trabajo)
    dias_totales = math.ceil(total_tomas / max_tomas_dia)
      
    if verbose: 
        print(f'{len(datos_trabajo.loc[0])-1} Actores')
        print(f'{total_tomas} Tomas')
        print(f'Máximo {max_tomas_dia} tomas por día, mínimo {dias_totales} días\n')
    
    # Array para guardar qué día le toca a cada toma
    plan_final = np.zeros(total_tomas)
    
    # Vamos rellenando los días uno por uno
    for dia_actual in range(1, dias_totales + 1):
        scores_parecidos = np.zeros(len(datos_trabajo))
    
        # 1. Buscamos la toma con más actores para que sea nuestra "ancla" del día
        indice_toma_ancla = datos_trabajo['Total'].idxmax()
        plan_final[indice_toma_ancla] = dia_actual
        # Marcamos esta toma para no volver a usarla
        scores_parecidos[datos_trabajo.index.get_loc(indice_toma_ancla)] = -999
    
        # 2. Calculamos cuántos actores comparten las demás tomas con nuestra ancla
        for posicion, indice_fila in enumerate(datos_trabajo.index):
            if indice_fila != indice_toma_ancla:
                # np.all comprueba qué actores salen en AMBAS tomas a la vez
                scores_parecidos[posicion] = np.sum(np.all(datos_trabajo.loc[[indice_fila, indice_toma_ancla], list(datos_trabajo.columns[:-1])], axis=0))
    
        # 3. Ordenamos las tomas por compatibilidad y cogemos las mejores
        datos_trabajo['Compatibilidad'] = scores_parecidos
        tomas_ordenadas = datos_trabajo.sort_values('Compatibilidad', ascending=False)
    
        # Seleccionamos las k-1 tomas más parecidas para agruparlas
        tomas_a_agrupar = tomas_ordenadas[:min(max_tomas_dia - 1, len(tomas_ordenadas))].index
        plan_final[tomas_a_agrupar] = dia_actual
    
        # 4. Quitamos las tomas ya asignadas de la lista de pendientes
        datos_trabajo.drop(tomas_a_agrupar, axis=0, inplace=True)
        datos_trabajo.drop(indice_toma_ancla, axis=0, inplace=True)

    # Convertimos a lista y calculamos el coste final con nuestra función
    plan_final = list(plan_final)
    # Nota: calcular_coste_rodaje debe estar definida previamente
    coste_final = calcular_coste_rodaje(plan_final, matriz_asistencia, verbose=False)
    
    if verbose: print(f'Solución voraz con coste {coste_final}: {plan_final}')
    return plan_final, coste_final

In [87]:
# Cronómetro 
inicio_voraz = time.time()

# Ejecutamos la función voraz
_, coste_voraz = alg_voraz(data, 6, verbose=False)

# Calculamos y pintamos el tiempo y el resultado
tiempo_voraz = round(time.time() - inicio_voraz, 2)
print(f'Estrategia Voraz: coste {coste_voraz} en {tiempo_voraz} segundos')

Estrategia Voraz: coste 35 en 0.18 segundos


--- 

## (*)Calcula la complejidad del algoritmo

La complejidad es:$$\frac{1}{k} n^3 + \left(2+9+\log(n)+a\frac{k}{k}\right)n^2 + (1+10\frac{k}{k})n + 5$$
El término dominante $\frac{1}{k} n^3$: Representa el bucle principal donde recorremos los días y, dentro de cada día, evaluamos la compatibilidad de las tomas. En este caso hablamos de $O(n^3)$.

---

## Según el problema (y tenga sentido), diseña un juego de datos de entrada aleatorios

### Búsqueda Aleatoria

Ya vimos que la Fuerza Bruta es inviable aquí, y por otra parte la voracidad no asegura la solución más óptima. Ya que el enunciado pide "diseñar un juego de datos de entrada aleatorios", hemos llegado a la conclusión de que usar una exploración aleatoria podría ser una pista en este caso, por lo tanto vamos a proceder a aplicar el algoritmo de búsqueda aleatoria.

La lógica a seguir será: en lugar buscar todas las combinaciones, lanzamos X intentos al azar y nos quedamos con el mejor de ellos. Al ser probabilístico y heruístico, no podemos asegurar tampoco que es la solución más óptima.

In [88]:
def busqueda_aleatoria(intentos, data, max_tomas_dia, mostrar_info=True):
    # 1. ANALISIS DE ESCENARIO
    total_tomas = len(data)
    # Calculamos días necesarios redondeando hacia arriba
    dias_necesarios = math.ceil(total_tomas / max_tomas_dia)                
    
    if mostrar_info:
        # data.shape[1] nos da el numero de columnas (actores)
        print(f'{data.shape[1]} Actores totales')
        print(f'{total_tomas} Tomas por grabar')
        print(f'Límite: {max_tomas_dia} tomas/día -> Mínimo {dias_necesarios} días\n')

    # 2. GENERAR PLANTILLA DE DÍAS
    # Creamos una lista con los días [1,1,1, 2,2,2, ...]
    # 1. Creamos una lista vacía para empezar
    distribucion_base = []
    
    # 2. Bucle exterior: Recorre cada día necesario
    for dia in range(1, dias_necesarios + 1):
        
        # 3. Bucle interior: Añade el número del día actual 'max_tomas_dia' veces
        for _ in range(max_tomas_dia):
            distribucion_base.append(dia)
    
    # Ajustamos al tamaño real de tomas por si acaso
    distribucion_base = distribucion_base[:total_tomas]

    # 3. BUCLE DE EXPLORACIÓN
    coste_minimo = float('inf')
    mejor_plan = []
    
    for i in range(intentos):
        # Desordenamos los días para probar una mezcla nueva
        random.shuffle(distribucion_base)
        
        # Calculamos el coste del desorden actual
        # Asumo que 'funcion_pago' es tu función que calcula el coste
        coste_actual = calcular_coste_rodaje(distribucion_base, data, verbose=False)

        # Si es mejor que lo anterior, guardamos
        if coste_actual < coste_minimo:
            coste_minimo = coste_actual
            mejor_plan = distribucion_base.copy()
            if mostrar_info: 
                print(f'Intento {i+1}: Nuevo mejor coste {coste_minimo}')

    if mostrar_info: 
        print(f'\nFinalizado. Coste final: {coste_minimo}')
        
    return mejor_plan, coste_minimo

## Aplica el algoritmo al juego de datos generado

In [89]:
# --- CRONÓMETRO ---
inicio_aleatoria = time.time()

# --- EJECUCIÓN DE BÚSQUEDA ALEATORIA ---
# Usamos 10000 intentos como ejemplo
plan_aleatorio, coste_aleatorio = busqueda_aleatoria(1000, data, max_tomas_dia=6, mostrar_info=False)

# --- RESULTADOS ---
tiempo_aleatoria = round(time.time() - inicio_aleatoria, 2)
print(f'Estrategia Aleatoria: coste {coste_aleatorio} en {tiempo_aleatoria} segundos')

Estrategia Aleatoria: coste 33 en 3.38 segundos


Respuesta

Enumera las referencias que has utilizado(si ha sido necesario) para llevar a cabo el trabajo

Respuesta

Describe brevemente las lineas de como crees que es posible avanzar en el estudio del problema. Ten en cuenta incluso posibles variaciones del problema y/o variaciones al alza del tamaño

Respuesta

### Clasificación Integral de Algoritmos de Optimización

| Algoritmo | Categoría | Forma de Calcular | Calidad de Solución | Características Principales | Aplicaciones Comunes |
| :--- | :--- | :--- | :--- | :--- | :--- |
| **Simplex** | Programación Matemática | Determinista | Exacto | Se desplaza por la frontera de la región factible hasta encontrar el óptimo matemático. | Problemas de transporte, dieta y producción. |
| **Ramificación y Poda** | Búsqueda Informada | Determinista | Exacto | Divide el problema en subproblemas y descarta ramas que no pueden superar al mejor resultado hallado. | Agente Viajero (TSP) y problemas NP-completos. |
| **Programación Dinámica** | Programación Dinámica | Determinista | Exacto | Divide el problema en subproblemas superpuestos y almacena sus soluciones en tablas para evitar cálculos repetidos. | Problema de la mochila y rutas de coste mínimo. |
| **Vuelta Atrás (Backtracking)** | Búsqueda Sistemática | Determinista | Exacto | Explora un árbol de expansión de manera exhaustiva y retrocede cuando una rama deja de ser prometedora. | Problema de las N-reinas y laberintos. |
| **Ordenación (p.ej. Merge Sort)** | Algoritmos de Ordenación | Determinista | Exacto | Sigue pasos finitos y precisos para organizar una lista bajo un criterio estricto. | Gestión de bases de datos y listas. |
| **Algoritmos Voraces (Greedy)** | Técnica de Diseño | Determinista | Heurístico / Aproximado | Elige en cada etapa la decisión óptima local sin considerar las consecuencias futuras; no garantiza el óptimo global. | Árboles de recubrimiento (Prim/Kruskal) y cambio de monedas. |
| **Multiarranque (Multi-start)** | Metaheurística Constructiva | Probabilista / Determinista | Heurístico | Repite el proceso de generar una solución inicial y aplicar búsqueda local para explorar diversas regiones y evitar mínimos locales. | Optimización global y problemas de rutas complejos. |
| **Búsqueda Aleatoria** | Metaheurística | Probabilista | Heurístico | Genera soluciones totalmente al azar en cada iteración y devuelve la mejor encontrada. | Exploración global en espacios de búsqueda desconocidos. |
| **Búsqueda Local** | Metaheurística | Determinista / Probabilista | Heurístico | Parte de una solución inicial y busca mejorarla explorando su vecindad inmediata; tiene riesgo de estancarse en óptimos locales. | Refinamiento de soluciones en problemas de rutas. |
| **GRASP** | Metaheurística | Probabilista | Heurístico | Combina una fase constructiva voraz-aleatoria con una fase de mejora mediante búsqueda local. | Problemas de logística y recubrimiento de conjuntos. |
| **Búsqueda Tabú** | Metaheurística | Determinista / Probabilista | Heurístico | Utiliza una memoria adaptativa (lista tabú) para prohibir movimientos recientes y forzar la exploración de nuevas áreas. | Programación de horarios (scheduling) y secuenciación industrial. |
| **Recocido Simulado** | Metaheurística | Probabilista | Heurístico | Acepta soluciones peores con una probabilidad que disminuye con el tiempo, imitando el enfriamiento de metales. | Diseño de circuitos integrados y metalurgia computacional. |
| **Algoritmos Genéticos** | Poblacional | Probabilista | Heurístico | Se basan en mecanismos de selección natural, cruce y mutación aplicados a una población de soluciones. | Bioinformática y optimización de funciones complejas. |
| **Colonia de Hormigas (ACO)** | Poblacional | Probabilista | Heurístico | Múltiples agentes depositan feromonas en las rutas para guiar probabilísticamente hacia la mejor solución compartida. | Optimización de redes de transporte y telecomunicaciones. |
| **Descenso del Gradiente** | Optimización Continua | Determinista | Exacto (en funciones convexas) | Algoritmo iterativo que minimiza una función continua siguiendo la dirección opuesta al gradiente. | Entrenamiento de Redes Neuronales y Machine Learning. |
| **Dijkstra** | Búsqueda en Grafos | Determinista | Exacto | Encuentra el camino más corto desde un nodo origen al resto, siempre que no haya pesos negativos. | Navegadores GPS y protocolos de red. |